In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import glob
import math

# Loading Data

In [121]:
testRunFolderName = "TestRun5/AfterCodeEdit"
coordFileType = "Unity"
trialNum = 2

In [122]:
def getFilePath(fileNamePattern: str, trialNumber = 0) -> str:
    folderPath = os.path.join(os.getcwd(), testRunFolderName)
    pattern = os.path.join(folderPath, fileNamePattern)
    # file_path = os.path.join(folderPath, file_name)
    matching_files = glob.glob(pattern)
    matching_files.sort()

    if matching_files:
        file_path = matching_files[trialNumber]
    else:
        raise FileNotFoundError("No file found")
    
    return file_path

In [123]:
csvFileName = coordFileType + "Coordinates_Trial_" + str(trialNum) + ".csv"
csvFilePath = getFilePath(csvFileName)

triggeredEventsFileName = "TriggeredEvents_Trial_" + str(trialNum) + ".csv"
triggeredEventsFilePath = getFilePath(triggeredEventsFileName)

In [124]:
pose_df = pd.read_csv(csvFilePath)

timeStamp_df = pd.read_csv(triggeredEventsFilePath)

# Filtering Data

In [125]:
def get3dPointDistance(pose1, pose2) -> float:
    x = math.pow((pose1[0] - pose2[0]), 2)
    y = math.pow((pose1[1] - pose2[1]), 2)
    z = math.pow((pose1[2] - pose2[2]), 2)
    distance = math.sqrt(x + y + z)
    return distance

## Add waypoints to list

In [126]:
waypointPoses = []
for rowNum in range(10):
    AtWaypoint1 = timeStamp_df["AtWaypoint Time"].loc[timeStamp_df.index[rowNum]]
    LeavingWaypoint1 = timeStamp_df["LeavingWaypoint Time"].loc[timeStamp_df.index[rowNum]]
    mask = (pose_df["Timestamp"] >= AtWaypoint1) & (pose_df["Timestamp"] <= LeavingWaypoint1)

    poseData_df = pose_df[mask]
    # samplesCount = poseData_df.shape[0]
    # print("Valid Times: " + str(samplesCount))

    waypointPoses.append(poseData_df)

## Get Centroids

In [127]:
waypointCentroids = []

for wpData_df in waypointPoses:
    centroid = (wpData_df["X"].mean(), wpData_df["Y"].mean(), wpData_df["Z"].mean())
    print("%.3f , %.3f, %.3f" % (centroid[0], centroid[1], centroid[2]))
    waypointCentroids.append(centroid)

13.561 , -0.794, 10.647
15.444 , -0.779, 12.301
17.310 , -0.750, 13.959
19.215 , -0.729, 15.637
21.108 , -0.709, 17.284
22.988 , -0.681, 18.926
24.893 , -0.664, 20.575
26.764 , -0.643, 22.174
28.621 , -0.628, 23.850
30.487 , -0.596, 25.475


## Distance First and Last

In [128]:
pointDistance = get3dPointDistance(waypointCentroids[0], waypointCentroids[9])
error = 22.5 - pointDistance
print("Distance: %.4f m" % pointDistance)
print("Error: %.4f m" % error)

Distance: 22.5042 m
Error: -0.0042 m


In [129]:
test_df = waypointPoses[0]
samplesCount = test_df.shape[0]
print("Valid Times: " + str(samplesCount))
# print(test_df.head(10))

# Statistics
centroid = (test_df["X"].mean(), test_df["Y"].mean(), test_df["Z"].mean())
print("%.3f , %.3f, %.3f" % (centroid[0], centroid[1], centroid[2]))

distancesData = []
for rowNum in range(samplesCount):
    sampleX = test_df["X"].loc[test_df.index[rowNum]]
    sampleY = test_df["Y"].loc[(test_df.index[rowNum])]
    sampleZ = test_df["Z"].loc[(test_df.index[rowNum])]
    samplePose = (sampleX, sampleY, sampleZ)
    # print("%.3f , %.3f, %.3f" % (samplePose[0], samplePose[1], samplePose[2]))

    distance = get3dPointDistance(samplePose, centroid)
    distancesData.append(distance)
    # print("%.4f" % (distance))

distancesSeries = pd.Series(distancesData)
distancesSeries.describe()

Valid Times: 67
13.561 , -0.794, 10.647


count    67.000000
mean      0.013617
std       0.011183
min       0.005657
25%       0.007392
50%       0.008258
75%       0.014052
max       0.046066
dtype: float64

In [130]:
# distancesSeries.plot.hist(stacked=True, bins=50)